# BUSI Dataset Preprocessing

Este notebook executa o pré-processamento do dataset BUSI passo a passo, de forma transparente e segura.

## Checklist de pré-requisitos

Antes de executar, confirme:
- [ ] Ambiente virtual ativo (`source .venv/bin/activate`)
- [ ] Dependências instaladas (`uv pip install -r requirements.txt`)
- [ ] Dataset original em `data/Dataset_BUSI_with_GT/` com subpastas `benign/`, `malignant/`, `normal/`
- [ ] Arquivo `data/mapping_curated_BUSI.csv` presente (necessário se `CURATED=True`)

## O que este notebook faz

1. Valida imports e ambiente
2. Valida estrutura de pastas e arquivos
3. Configura parâmetros
4. Carrega dataframes por classe
5. Identifica imagens com múltiplas máscaras
6. Cria diretórios de saída
7. Redimensiona e combina imagens/máscaras
8. Gera o CSV de mapeamento com metadados
9. Exibe resumo final e próximos passos

## Célula 1 — Validação de ambiente e imports

In [1]:
import sys
import importlib

print(f"Python: {sys.version}")

required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'cv2': 'opencv-python',
    'pathlib': 'pathlib (stdlib)',
}

all_ok = True
for module, pkg_name in required_packages.items():
    try:
        importlib.import_module(module)
        print(f"  [OK] {pkg_name}")
    except ImportError:
        print(f"  [MISSING] {pkg_name} — instale com: uv pip install {pkg_name}")
        all_ok = False

if not all_ok:
    raise EnvironmentError("Dependências faltando. Instale antes de continuar.")
else:
    print("\nAmbiente OK.")

Python: 3.10.20 (main, Apr  7 2026, 20:44:18) [Clang 22.1.1 ]
  [OK] numpy
  [OK] pandas
  [OK] opencv-python
  [OK] pathlib (stdlib)

Ambiente OK.


## Célula 2 — Imports

In [2]:
import os
from pathlib import Path
from typing import List, Dict, Tuple

import numpy as np
import pandas as pd
import cv2

print(f"numpy  {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"cv2    {cv2.__version__}")

numpy  1.23.5
pandas 1.5.0
cv2    4.7.0


## Célula 3 — Parâmetros de configuração

Edite aqui antes de executar as células seguintes.

In [3]:
# Diretório raiz do projeto (relativo ao local de execução do notebook)
PROJECT_ROOT = Path("../")

ROOT_DATA = PROJECT_ROOT / "data"
INPUT_FOLDER = "Dataset_BUSI_with_GT"
CLASS_NAMES = ["benign", "malignant", "normal"]

RESIZE_DIMENSIONS = (128, 128)
OUTPUT_FOLDER = "Dataset_BUSI_with_GT_128"

# Modo curado: mantém apenas as imagens do CSV de mapeamento curado
CURATED = True
CURATED_MAPPING_FILE = ROOT_DATA / "mapping_curated_BUSI.csv"
CURATED_OUTPUT_FOLDER = "Curated_BUSI_128"

input_path = ROOT_DATA / INPUT_FOLDER
output_folder_name = CURATED_OUTPUT_FOLDER if CURATED else OUTPUT_FOLDER
output_path = ROOT_DATA / output_folder_name

print(f"Input  : {input_path.resolve()}")
print(f"Output : {output_path.resolve()}")
print(f"Curated: {CURATED}")
print(f"Resize : {RESIZE_DIMENSIONS}")

Input  : /home/lucas/fed_multi_task_breast_cancer/data/Dataset_BUSI_with_GT
Output : /home/lucas/fed_multi_task_breast_cancer/data/Curated_BUSI_128
Curated: True
Resize : (128, 128)


## Célula 4 — Validação de estrutura de pastas e arquivos

In [5]:

import shutil

class_subfolders_missing = [
    cls for cls in CLASS_NAMES
    if not (input_path / cls).exists()
]

if not class_subfolders_missing:
    print("Estrutura de classes já existe. Nenhuma ação necessária.")
else:
    print(f"Subpastas ausentes: {class_subfolders_missing}")

    # Verificar se há dados pré-processados em images/ que possam ser usados
    processed_images_dir = input_path / "images"
    processed_masks_dir  = input_path / "masks"

    if not processed_images_dir.exists() or not list(processed_images_dir.glob("*.png")):
        print("\n[ERRO] Nenhum dado encontrado em images/ para recuperar a estrutura.")
        print("Baixe o dataset BUSI original em: https://scholar.cu.edu.eg/?q=afahmy/pages/dataset")
        print("E extraia de forma que a estrutura seja:")
        for cls in CLASS_NAMES:
            print(f"  {input_path / cls}/")
        raise FileNotFoundError("Dataset BUSI original não encontrado.")

    # Recriar subpastas de classe com a nomenclatura original do BUSI
    # benign_id_100.png      → benign/benign (100).png
    # benign_id_100_mask.png → benign/benign (100)_mask.png
    print(f"\nEncontrados arquivos pré-processados em '{processed_images_dir}'. Recriando estrutura...")

    for cls in CLASS_NAMES:
        (input_path / cls).mkdir(exist_ok=True)

    img_count  = {cls: 0 for cls in CLASS_NAMES}
    skip_count = 0

    for img_file in sorted(processed_images_dir.glob("*.png")):
        parts = img_file.stem.split("_id_")   # ["benign", "100"]
        if len(parts) != 2 or parts[0] not in CLASS_NAMES:
            skip_count += 1
            continue
        cls, id_str = parts
        dest_img  = input_path / cls / f"{cls} ({id_str}).png"
        dest_mask = input_path / cls / f"{cls} ({id_str})_mask.png"
        shutil.copy2(str(img_file), str(dest_img))

        mask_file = processed_masks_dir / f"{cls}_id_{id_str}_mask.png"
        if mask_file.exists():
            shutil.copy2(str(mask_file), str(dest_mask))

        img_count[cls] += 1

    print("\nEstrutura recriada com sucesso:")
    for cls in CLASS_NAMES:
        print(f"  {cls:12s} — {img_count[cls]} imagens copiadas → {input_path / cls}")
    if skip_count:
        print(f"  ({skip_count} arquivo(s) ignorado(s) por não seguir o padrão {{classe}}_id_{{n}}.png)")


Subpastas ausentes: ['benign', 'malignant', 'normal']

Encontrados arquivos pré-processados em '../data/Dataset_BUSI_with_GT/images'. Recriando estrutura...

Estrutura recriada com sucesso:
  benign       — 222 imagens copiadas → ../data/Dataset_BUSI_with_GT/benign
  malignant    — 164 imagens copiadas → ../data/Dataset_BUSI_with_GT/malignant
  normal       — 64 imagens copiadas → ../data/Dataset_BUSI_with_GT/normal


In [6]:
errors = []

# 1. Pasta de entrada
if not input_path.exists():
    errors.append(f"[MISSING] Pasta de entrada não encontrada: {input_path.resolve()}\n"
                  "  -> Baixe o dataset BUSI em https://scholar.cu.edu.eg/?q=afahmy/pages/dataset")
else:
    print(f"[OK] Pasta de entrada: {input_path.resolve()}")
    for cls in CLASS_NAMES:
        cls_path = input_path / cls
        if not cls_path.exists():
            errors.append(f"[MISSING] Subpasta de classe ausente: {cls_path}")
        else:
            pngs = list(cls_path.glob("*.png"))
            print(f"  [OK] {cls:12s} — {len(pngs)} arquivos .png")
            if len(pngs) == 0:
                errors.append(f"[EMPTY] {cls_path} não contém arquivos .png")

# 2. CSV curado
if CURATED:
    if not CURATED_MAPPING_FILE.exists():
        errors.append(f"[MISSING] Arquivo de mapeamento curado: {CURATED_MAPPING_FILE.resolve()}")
    else:
        curated_df = pd.read_csv(CURATED_MAPPING_FILE, sep=';')
        required_cols = {'class', 'id'}
        missing_cols = required_cols - set(curated_df.columns)
        if missing_cols:
            errors.append(f"[INVALID] CSV curado sem colunas obrigatórias: {missing_cols}")
        else:
            print(f"[OK] Mapeamento curado: {len(curated_df)} linhas, colunas: {list(curated_df.columns)}")

if errors:
    print("\n=== ERROS ENCONTRADOS ===")
    for e in errors:
        print(e)
    raise FileNotFoundError("Corrija os erros acima antes de continuar.")
else:
    print("\nEstrutura de dados validada com sucesso.")

[OK] Pasta de entrada: /home/lucas/fed_multi_task_breast_cancer/data/Dataset_BUSI_with_GT
  [OK] benign       — 444 arquivos .png
  [OK] malignant    — 328 arquivos .png
  [OK] normal       — 128 arquivos .png
[OK] Mapeamento curado: 450 linhas, colunas: ['class', 'id']

Estrutura de dados validada com sucesso.


## Célula 5 — Carregar dataframes por classe

In [7]:
def load_class_dataframe(class_path: Path, class_name: str) -> pd.DataFrame:
    files = [f for f in sorted(os.listdir(class_path)) if f.endswith(".png")]
    ids = [
        f.replace(".png", "").split(" ")[-1].split("_")[0]
        .replace("(", "").replace(")", "")
        for f in files
    ]
    types = ["mask" if "mask" in f else "img" for f in files]
    df = pd.DataFrame({"class": [class_name] * len(ids), "ids": ids, "type": types})
    print(f"  {class_name:12s} — {len(df[df['type']=='img'])} imagens, {len(df[df['type']=='mask'])} máscaras")
    return df

print("Carregando dataframes por classe:")
df_classes = [load_class_dataframe(input_path / cls, cls) for cls in CLASS_NAMES]

# Visualização rápida
for df, cls in zip(df_classes, CLASS_NAMES):
    display(df.head(5))

Carregando dataframes por classe:
  benign       — 222 imagens, 222 máscaras
  malignant    — 164 imagens, 164 máscaras
  normal       — 64 imagens, 64 máscaras


,class,ids,type
0,benign,100,img
1,benign,100,mask
2,benign,101,img
3,benign,101,mask
4,benign,102,img


,class,ids,type
0,malignant,1,img
1,malignant,1,mask
2,malignant,100,img
3,malignant,100,mask
4,malignant,101,img


,class,ids,type
0,normal,10,img
1,normal,10,mask
2,normal,100,img
3,normal,100,mask
4,normal,101,img


## Célula 6 — Identificar imagens com múltiplas máscaras

In [8]:
def get_mask_counts(df: pd.DataFrame, class_name: str) -> List[int]:
    counts = df.groupby("ids").apply(lambda x: sum(x['type'] == 'mask')).to_dict()
    multi = [int(k) for k, v in counts.items() if v > 1]
    print(f"  {class_name:12s} — {len(multi)} imagens com múltiplas máscaras")
    return multi

print("Imagens com múltiplas máscaras (serão combinadas):")
multi_masks_per_class = [
    get_mask_counts(df, cls)
    for df, cls in zip(df_classes, CLASS_NAMES)
]

Imagens com múltiplas máscaras (serão combinadas):
  benign       — 0 imagens com múltiplas máscaras
  malignant    — 0 imagens com múltiplas máscaras
  normal       — 0 imagens com múltiplas máscaras


## Célula 7 — Carregar IDs do mapeamento curado

In [9]:
if CURATED:
    curated_mapping = pd.read_csv(CURATED_MAPPING_FILE, sep=';')
    curated_ids_dict = {
        cls: curated_mapping[curated_mapping['class'] == cls]['id'].astype(int).tolist()
        for cls in CLASS_NAMES
    }
    for cls, ids in curated_ids_dict.items():
        print(f"  {cls:12s} — {len(ids)} IDs no mapeamento curado")
else:
    curated_ids_dict = {cls: None for cls in CLASS_NAMES}
    print("Modo não-curado: todas as imagens serão processadas.")

  benign       — 222 IDs no mapeamento curado
  malignant    — 164 IDs no mapeamento curado
  normal       — 64 IDs no mapeamento curado


## Célula 8 — Criar diretórios de saída

In [10]:
(output_path / "images").mkdir(parents=True, exist_ok=True)
(output_path / "masks").mkdir(parents=True, exist_ok=True)

print(f"[OK] {output_path / 'images'}")
print(f"[OK] {output_path / 'masks'}")

[OK] ../data/Curated_BUSI_128/images
[OK] ../data/Curated_BUSI_128/masks


## Célula 9 — Redimensionar e salvar imagens/máscaras

In [11]:
def combine_and_resize_images(
    class_name: str,
    class_ids: List[str],
    multi_mask_ids: List[int],
    path: Path,
    out_path: Path,
    curated_ids: List[int] = None
) -> int:
    saved = 0
    for j in set(class_ids):
        j_int = int(j)
        if curated_ids is not None and j_int not in curated_ids:
            continue

        img_file = path / class_name / f"{class_name} ({j}).png"
        if not img_file.exists():
            print(f"  [WARN] Imagem não encontrada: {img_file}")
            continue

        img = cv2.imread(str(img_file), 0)

        mask_files = [f"{class_name} ({j})_mask.png"]
        if j_int in multi_mask_ids:
            mask_files.append(f"{class_name} ({j})_mask_1.png")

        total_mask = sum(
            cv2.imread(str(path / class_name / mf), 0)
            for mf in mask_files
            if (path / class_name / mf).exists()
        )

        img_resized = cv2.resize(img, RESIZE_DIMENSIONS, interpolation=cv2.INTER_NEAREST)
        mask_resized = cv2.resize(total_mask, RESIZE_DIMENSIONS, interpolation=cv2.INTER_NEAREST)

        cv2.imwrite(str(out_path / "images" / f"{class_name}_id_{j}.png"), img_resized)
        cv2.imwrite(str(out_path / "masks" / f"{class_name}_id_{j}_mask.png"), mask_resized)
        saved += 1

    return saved

print("Processando imagens e máscaras:")
for i, (df, cls) in enumerate(zip(df_classes, CLASS_NAMES)):
    n = combine_and_resize_images(
        cls,
        df['ids'].tolist(),
        multi_masks_per_class[i],
        input_path,
        output_path,
        curated_ids_dict.get(cls)
    )
    print(f"  {cls:12s} — {n} imagens salvas")

print("\nProcessamento concluído.")

Processando imagens e máscaras:
  benign       — 222 imagens salvas
  malignant    — 164 imagens salvas
  normal       — 64 imagens salvas

Processamento concluído.


## Célula 10 — Gerar CSV de mapeamento

In [12]:
img_paths = sorted((output_path / "images").glob("*.png"))

project_root = PROJECT_ROOT.resolve()
df_mapping = pd.DataFrame({"img_path": [str(p.resolve().relative_to(project_root)) for p in img_paths]})
df_mapping['mask_path'] = (
    df_mapping['img_path']
    .str.replace("images", "masks", regex=False)
    .str.replace(".png", "_mask.png", regex=False)
)
df_mapping['class'] = df_mapping['img_path'].apply(lambda x: Path(x).stem.split('_')[0])
df_mapping['id'] = df_mapping['img_path'].apply(lambda x: int(Path(x).stem.split('_')[-1]))

print(f"Total de imagens mapeadas: {len(df_mapping)}")
display(df_mapping.head())

Total de imagens mapeadas: 450


,img_path,mask_path,class,id
0,../data/Curated_BUSI_128/images/benign_id_100.png,../data/Curated_BUSI_128/masks/benign_id_100_m...,benign,100
1,../data/Curated_BUSI_128/images/benign_id_101.png,../data/Curated_BUSI_128/masks/benign_id_101_m...,benign,101
2,../data/Curated_BUSI_128/images/benign_id_102.png,../data/Curated_BUSI_128/masks/benign_id_102_m...,benign,102
3,../data/Curated_BUSI_128/images/benign_id_103.png,../data/Curated_BUSI_128/masks/benign_id_103_m...,benign,103
4,../data/Curated_BUSI_128/images/benign_id_104.png,../data/Curated_BUSI_128/masks/benign_id_104_m...,benign,104


## Célula 11 — Adicionar metadados (dimensões, pixels de tumor, bounding box)

In [13]:
def count_pixels(seg: np.ndarray) -> Dict[int, int]:
    unique, counts = np.unique(seg, return_counts=True)
    return dict(zip(unique, counts))

def size_tumor(seg: np.ndarray) -> Tuple[int, int, int, int, int, int]:
    y_idx, x_idx = np.nonzero(seg != 0)
    if len(y_idx) == 0 or len(x_idx) == 0:
        return 0, 0, 0, 0, 0, 0
    ymin = max(0, int(np.min(y_idx)))
    xmin = max(0, int(np.min(x_idx)))
    ymax = int(np.max(y_idx) + 1)
    xmax = int(np.max(x_idx) + 1)
    return ymax, ymin, xmax, xmin, ymax - ymin, xmax - xmin

dims1, dims2, tumor_pixels = [], [], []
ymaxl, yminl, xmaxl, xminl, y_sizel, x_sizel = [], [], [], [], [], []

for img_p, mask_p in zip(df_mapping['img_path'], df_mapping['mask_path']):
    img = cv2.imread(img_p, 0)
    dims1.append(img.shape[0])
    dims2.append(img.shape[1])

    mask = cv2.imread(mask_p, 0)
    counting = count_pixels(mask)
    tumor_pixels.append(counting.get(255, 0))

    ymax, ymin, xmax, xmin, y_size, x_size = size_tumor(mask)
    ymaxl.append(ymax); yminl.append(ymin)
    xmaxl.append(xmax); xminl.append(xmin)
    y_sizel.append(y_size); x_sizel.append(x_size)

df_mapping['dim1'] = dims1
df_mapping['dim2'] = dims2
df_mapping['tumor_pixels'] = tumor_pixels
df_mapping['y_max'] = ymaxl
df_mapping['y_min'] = yminl
df_mapping['x_max'] = xmaxl
df_mapping['x_min'] = xminl
df_mapping['y_size'] = y_sizel
df_mapping['x_size'] = x_sizel

df_mapping = df_mapping.sort_values(by=['class', 'id']).reset_index(drop=True)
display(df_mapping.head())

,img_path,mask_path,class,id,dim1,dim2,tumor_pixels,y_max,y_min,x_max,x_min,y_size,x_size
0,../data/Curated_BUSI_128/images/benign_id_20.png,../data/Curated_BUSI_128/masks/benign_id_20_ma...,benign,20,128,128,704,48,28,95,49,20,46
1,../data/Curated_BUSI_128/images/benign_id_22.png,../data/Curated_BUSI_128/masks/benign_id_22_ma...,benign,22,128,128,867,84,51,75,39,33,36
2,../data/Curated_BUSI_128/images/benign_id_23.png,../data/Curated_BUSI_128/masks/benign_id_23_ma...,benign,23,128,128,675,82,55,88,56,27,32
3,../data/Curated_BUSI_128/images/benign_id_24.png,../data/Curated_BUSI_128/masks/benign_id_24_ma...,benign,24,128,128,344,79,60,85,63,19,22
4,../data/Curated_BUSI_128/images/benign_id_26.png,../data/Curated_BUSI_128/masks/benign_id_26_ma...,benign,26,128,128,2112,66,22,76,13,44,63


## Célula 12 — Salvar CSV e exibir resumo

In [14]:
mapping_csv_path = output_path / "mapping.csv"
df_mapping.to_csv(str(mapping_csv_path), index=False)

print(f"[OK] Mapeamento salvo em: {mapping_csv_path.resolve()}")
print(f"\nTotal de imagens processadas: {len(df_mapping)}")
for cls in CLASS_NAMES:
    n = len(df_mapping[df_mapping['class'] == cls])
    print(f"  {cls.capitalize():12s}: {n} imagens")

[OK] Mapeamento salvo em: /home/lucas/fed_multi_task_breast_cancer/data/Curated_BUSI_128/mapping.csv

Total de imagens processadas: 450
  Benign      : 222 imagens
  Malignant   : 164 imagens
  Normal      : 64 imagens


## Próximos passos

O pré-processamento foi concluído. Para continuar:

1. **Edite `src/config.yaml`** — atualize o campo `data.input_img` com o caminho do dataset gerado:
   ```yaml
   data:
     input_img: data/Curated_BUSI_128   # ou Dataset_BUSI_with_GT_128 se CURATED=False
   ```

2. **Execute o treinamento**:
   ```bash
   python -m src.training_multitask
   ```

   Ou para segmentação/classificação separada:
   ```bash
   python -m src.training_segmentation
   python -m src.training_classification
   ```